In [ ]:
#import packages
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.neural_network import MLPRegressor # Install ANN model 
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from ta import add_all_ta_features # Library that does financial technical analysis 

#to plot within notebook
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

#for normalizing data
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))

In [ ]:
# Importing the training set
df = pd.read_csv('Data/BANK-OF-AFRICA.csv', index_col="Date", parse_dates=True)

# Add all technical analysis to the dataframe we've already loaded
df = add_all_ta_features(df, "Open", "High", "Low", "Close", "Volume", fillna=True)

target_col = 'Close'

In [ ]:
#plot
plt.figure(figsize=(16,8))
plt.plot(df[target_col], label='Close Price history')

In [ ]:
from sklearn.model_selection import train_test_split
# Split data into testing and training sets
X = df.drop(target_col, axis=1)
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, shuffle=False)

train = X_train
train[target_col] = y_train
test = X_test
test[target_col] = y_test

# Linear Regression

In [ ]:
# Regreesion Function
def LinearRegression_fnc(X_train, y_train, X_test, y_test):
    # inputs: x train data, y train data, x test data, y test data (all dataframe's)
    # output: the predicted values for the test data (list)
    
    #scaling data
    scaler.fit(X_train)
    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)
    
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    lr_pred = lr.predict(X_test)
    
    lr_MSE = mean_squared_error(y_test, lr_pred)
    lr_MAE = mean_absolute_error(y_test, lr_pred)
    lr_RMS = np.sqrt(np.mean(np.power((np.array(y_test)-np.array(lr_pred)),2)))
    lr_R2 = lr.score(X_test, y_test)
    lr_RR2 = r2_score(y_test, lr_pred)
    print('Linear Regression R2: {}'.format(lr_R2))
    print("Coefficient of Determination: {}".format(lr_RR2))
    print('Linear Regression Mean Absolute Error: {}'.format(lr_MAE))
    print('Linear Regression MSE: {}'.format(lr_MSE))
    print('Linear Regression RMS: {}'.format(lr_RMS))

    return lr_pred

In [ ]:
lr_pred = LinearRegression_fnc(X_train, y_train, X_test, y_test)

# ANN

In [ ]:
# ANN Function 
def ANN_func(X_train, y_train, X_test, y_test):   
    # Scaling data
    # scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    MLP = MLPRegressor(random_state=1, max_iter=1000, hidden_layer_sizes = (100,), activation = 'identity',learning_rate = 'adaptive').fit(X_train_scaled, y_train)
    MLP_pred = MLP.predict(X_test_scaled)
    
    MLP_MSE = mean_squared_error(y_test, MLP_pred)
    MLP_MAE = mean_absolute_error(y_test, MLP_pred)
    MLP_RMS = np.sqrt(np.mean(np.power((np.array(y_test)-np.array(MLP_pred)),2)))
    MLP_R2 = MLP.score(X_test_scaled, y_test)
    MLP_RR2 = r2_score(y_test, MLP_pred)
    print('Muli-layer Perceptron R2 Test: {}'.format(MLP_R2))
    print("Coefficient of Determination: {}".format(MLP_RR2))
    print('Multi-layer Perceptron Mean Absolute Error: {}'.format(MLP_MAE))
    print('Multi-layer Perceptron MSE: {}'.format(MLP_MSE))
    print('Multi-layer Perceptron RMS: {}'.format(MLP_RMS))

    return MLP_pred

In [ ]:
MLP_pred = ANN_func(X_train, y_train, X_test, y_test)

In [ ]:
# Function to make the plots
def PlotModelResults(train, test, pred):
    test['Predictions'] = 0
    test['Predictions'] = pred

    test.index = X_test.index
    train.index = X_train.index

    plt.figure(figsize=(16,8))
#     plt.plot(train['Close'], label='Train Actual')
    plt.plot(test['Close'], label='Test Actual')
    plt.plot(test['Predictions'], label='Our Prediction')
    plt.title('Stock Price')
    plt.xlabel('Time [days]')
    plt.ylabel('Price')
    plt.legend(loc='best')

In [ ]:
PlotModelResults(train, test, lr_pred)

In [ ]:
PlotModelResults(train, test, MLP_pred)